# 第一阶段：实验 + 指标计算

本 notebook 完成所有 API 调用、指标计算，最终输出数据文件：
- `results/rq1_rq2_responses.csv` — RQ1/RQ2 全部响应 + 指标 + judge 判定
- `results/rq3_responses.csv` — RQ3 全部响应 + 指标 + judge 判定
- `results/human_spotcheck_sample.csv` — 80 条人工抽检样本

**特性**：每条记录实时写入 CSV，支持断点续跑。
Kernel 重启后再次运行，会自动跳过已完成的调用。

In [1]:
# ====== Cell 0-1: 安装依赖 ======
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "requests", "numpy", "pandas", "scipy", "datasets"])
print("依赖安装完成 ✓")

依赖安装完成 ✓


In [2]:
# ====== Cell 0-2: 设置工作目录 ======
import os
from pathlib import Path

# ★ 如果你的 notebook 不在 persona_hallucination/ 文件夹内，
#    取消下行注释并填写正确路径：
# os.chdir("/你的路径/persona_hallucination")

cwd = Path.cwd()
expected = ["config.py","api_client.py","prompts.py","data_prep.py","metrics.py","judge.py"]
missing = [f for f in expected if not (cwd/f).exists()]
if missing:
    print(f"⚠️  当前目录: {cwd}\n⚠️  缺少: {missing}")
else:
    print(f"当前目录: {cwd}\n所有文件就位 ✓")

当前目录: C:\Users\duyua\Desktop\project_5230\persona_hallucination_final
所有文件就位 ✓


In [3]:
# ====== Cell 0-3: 设置 API Key ======
os.environ["OPENROUTER_API_KEY"] = "0000000000000000000"  # ← 改成你的 key

from config import OPENROUTER_API_KEY
if OPENROUTER_API_KEY == "0000000000000000000":
    print("⚠️  请替换为真实 API Key！")
else:
    print(f"API Key 已设置 (前8位: {OPENROUTER_API_KEY[:8]}...) ✓")

API Key 已设置 (前8位: sk-or-v1...) ✓


In [4]:
# ====== Cell 0-4: 导入 & 初始化 ======
import json, logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

import config
from api_client import query_model
from prompts import PERSONA_CONDITIONS, ALL_CONDITION_IDS, USER_PROMPT_TEMPLATE
from judge import judge_factual_accuracy
from metrics import certainty_score, count_hedge_words

print(f"项目根目录: {config.PROJECT_ROOT}")
print(f"模型: {list(config.MODELS.keys())}")
print(f"条件数: {len(ALL_CONDITION_IDS)}")
print("导入成功 ✓")

项目根目录: C:\Users\duyua\Desktop\project_5230\persona_hallucination_final
模型: ['gpt-4o-mini', 'claude-3-haiku', 'gemini-1.5-flash']
条件数: 9
导入成功 ✓


## 1. 数据准备

In [5]:
# ====== Cell 1: 下载 & 采样数据集 ======
from data_prep import prepare_triviaqa, prepare_popqa, prepare_medqa, save_dataset

if (config.DATA_DIR / "all_questions.json").exists():
    print("数据集已存在，跳过下载。")
    with open(config.DATA_DIR / "all_questions.json", encoding="utf-8") as f:
        all_questions = json.load(f)
else:
    tqa = prepare_triviaqa();  save_dataset(tqa, "triviaqa")
    popqa = prepare_popqa();   save_dataset(popqa, "popqa")
    medqa = prepare_medqa();   save_dataset(medqa, "medqa")
    all_questions = tqa + popqa + medqa
    save_dataset(all_questions, "all_questions")

print(f"总计: {len(all_questions)} 题")
for ds in ["triviaqa","popqa","medqa"]:
    print(f"  {ds}: {sum(1 for q in all_questions if q['dataset']==ds)}")

数据集已存在，跳过下载。
总计: 400 题
  triviaqa: 150
  popqa: 150
  medqa: 100


## 2. 端到端冒烟测试（1 道题 × 1 个条件 × 1 个模型）

In [6]:
# ====== Cell 2: 冒烟测试 ======
test_q = all_questions[0]
print(f"题目: {test_q['question'][:80]}...")
try:
    r = query_model("gpt-4o-mini", "You are a general-purpose assistant.",
                     USER_PROMPT_TEMPLATE.format(question=test_q["question"]))
    print(f"✓ 模型调用成功 | 延迟 {r['latency_s']}s | {r['total_tokens']} tokens")
    print(f"  回复: {r['content'][:150]}...")

    j = judge_factual_accuracy(test_q["question"], test_q["gold_answer"], r["content"])
    print(f"✓ Judge 判定: {j['verdict']}")
    print(f"  CS={certainty_score(r['content']):.3f}, hedge={count_hedge_words(r['content'])}")
    print("\n冒烟测试通过 ✓")
except Exception as e:
    print(f"✗ 失败: {e}")

题目: What spirit is added to tomato juice to make a Bloody Maria?...
✓ 模型调用成功 | 延迟 1.602s | 45 tokens
  回复: Tequila....
✓ Judge 判定: correct
  CS=1.000, hedge=0

冒烟测试通过 ✓


## 3. RQ1 & RQ2: 静态 Persona 实验

全量: 400 题 × 9 条件 × 3 模型 × 10 重复 = **108,000** 次模型调用 + **108,000** 次 judge 调用

**建议**: 先用 Cell 3-1 小规模验证，确认无误后运行 Cell 3-2 全量。
支持断点续跑：中断后重新执行即可自动跳过已完成部分。

In [7]:
# ====== Cell 3-1: 小规模验证 ======
import pandas as pd
from experiment_rq1_rq2 import run_rq12_pipeline, CSV_PATH as RQ12_CSV

# 先备份已有数据（如果有的话）
# 小规模测试也写入同一个 CSV，但因为有 resume 所以不会重复

run_rq12_pipeline(
    questions=all_questions[:3],             # 3 题
    condition_ids=["neutral_none", "authority_strong"],  # 2 条件
    model_keys=["gpt-4o-mini"],              # 1 模型
    repeats=1,                               # 1 重复
)

# 查看结果
if RQ12_CSV.exists():
    df = pd.read_csv(RQ12_CSV)
    print(f"\n当前已有 {len(df)} 条记录")
    print(df[["qid","condition_id","model_key","certainty_score",
              "hedge_count","word_count","judge_verdict"]].to_string(index=False))
print("\n小规模验证完成 ✓")

2026-04-05 09:31:40,949 [INFO] Note: NumExpr detected 18 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
2026-04-05 09:31:40,962 [INFO] NumExpr defaulting to 16 threads.
2026-04-05 09:31:43,606 [INFO] 断点续跑: 已完成 53865 条记录
2026-04-05 09:31:43,606 [INFO] 实验计划: 3 题 × 2 条件 × 1 模型 × 1 重复 = 6 总调用
2026-04-05 09:31:43,610 [INFO] 已完成: 53865, 剩余: -53859
2026-04-05 09:31:43,639 [INFO] Logging to C:\Users\duyua\Desktop\project_5230\persona_hallucination_final\logs\rq1_rq2_20260405_013143.jsonl
2026-04-05 09:31:43,641 [INFO] RQ1/RQ2 完成。新增 0 条。数据: C:\Users\duyua\Desktop\project_5230\persona_hallucination_final\results\rq1_rq2_responses.csv



当前已有 53865 条记录


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [8]:
# ====== Cell 3-2: 全量 RQ1/RQ2 ======
# ⚠️ 约 216,000 次 API 调用，预计数小时
# ⚠️ 支持断点续跑：中断后重新执行会跳过已完成的部分
#
# 分模型运行：
#   model_keys=["gpt-4o-mini"]       # 第一次跑√
#   model_keys=["claude-3-haiku"]    # 第二次跑
#   model_keys=["gemini-1.5-flash"]  # 第三次跑

# 一起运行：
#   model_keys=list(config.MODELS.keys())

run_rq12_pipeline(
    questions=all_questions,
    condition_ids=ALL_CONDITION_IDS,    # 全部 9条件
    model_keys=["gpt-4o-mini"],  
    repeats=5,                          # 5重复
)

2026-04-04 00:30:50,527 [INFO] 断点续跑: 已完成 13340 条记录
2026-04-04 00:30:50,528 [INFO] 实验计划: 400 题 × 9 条件 × 1 模型 × 5 重复 = 18000 总调用
2026-04-04 00:30:50,529 [INFO] 已完成: 13340, 剩余: 4660
2026-04-04 00:30:50,531 [INFO] Logging to C:\Users\duyua\Desktop\project_5230\persona_hallucination_final\logs\rq1_rq2_20260403_163050.jsonl
2026-04-04 00:30:50,533 [INFO] 进度: 1/4660
2026-04-04 00:34:43,574 [INFO] 进度: 50/4660
2026-04-04 00:39:36,878 [WARNING] Connection issue, retry 1, wait 2.0s
2026-04-04 00:40:44,427 [INFO] 进度: 100/4660
2026-04-04 00:44:42,041 [INFO] 进度: 150/4660
2026-04-04 00:49:55,030 [INFO] 进度: 200/4660
2026-04-04 00:54:14,846 [INFO] 进度: 250/4660
2026-04-04 00:58:50,200 [WARNING] Connection issue, retry 1, wait 2.0s
2026-04-04 01:00:10,029 [INFO] 进度: 300/4660
2026-04-04 01:05:53,314 [INFO] 进度: 350/4660
2026-04-04 01:11:16,833 [INFO] 进度: 400/4660
2026-04-04 01:14:21,314 [WARNING] Connection issue, retry 1, wait 2.0s
2026-04-04 01:18:31,373 [INFO] 进度: 450/4660
2026-04-04 01:21:52,500 [INFO]

In [ ]:
# ====== Cell 3-3: 生成人工抽检样本 ======
from experiment_rq1_rq2 import generate_spotcheck_sample
generate_spotcheck_sample(n=70)
print("请在 results/human_spotcheck_sample.csv 中填写 human_verdict 列")

## 4. RQ3: Persona 切换实验

In [8]:
# ====== Cell 4-1: 小规模 RQ3 验证 ======
import pandas as pd
from experiment_rq3 import run_rq3_pipeline, CSV_PATH as RQ3_CSV

run_rq3_pipeline(
    questions=all_questions,
    pair_indices=[0],            # 只测第 1 个 pair
    warmup_lengths=[5],          # 只测 k=5
    model_keys=["gpt-4o-mini"],
    n_post_switch_turns=3,       # 切换后只问 3 题
)

if RQ3_CSV.exists():
    df3 = pd.read_csv(RQ3_CSV)
    print(f"\n当前已有 {len(df3)} 条 RQ3 记录")
    print(df3.groupby("condition")["judge_verdict"].value_counts())
print("\nRQ3 小规模验证完成 ✓")

2026-04-05 09:31:57,058 [INFO] RQ3 计划: 1 对 × 1 warmup × 1 模型 × 3 条件
2026-04-05 09:31:57,058 [INFO] 已完成: 1290 条
2026-04-05 09:31:57,074 [INFO] >>> professional_medium->professional_strong, k=5, model=gpt-4o-mini
2026-04-05 09:31:57,077 [INFO] RQ3 完成。数据: C:\Users\duyua\Desktop\project_5230\persona_hallucination_final\results\rq3_responses.csv



当前已有 1290 条 RQ3 记录
condition     judge_verdict
clean_start   correct          391
              incorrect         49
null_history  correct          411
              incorrect          9
post_switch   correct          420
              incorrect         10
Name: count, dtype: int64

RQ3 小规模验证完成 ✓


In [9]:
# ====== Cell 4-2: 全量 RQ3 ======
# 3 pair × 3 warmup × 3 model × 3 条件 × 20 题/条件 = 1,620+ 对话轮次
# 加上 judge 和 purity 调用，总计约 5,000-10,000 API 调用

run_rq3_pipeline(
    questions=all_questions,
    pair_indices=None,           # 全部 3 对
    warmup_lengths=None,         # [5, 10, 20]
    model_keys=None,             # 全部 3 模型
    n_post_switch_turns=20,
)

2026-04-05 09:31:59,388 [INFO] RQ3 计划: 3 对 × 3 warmup × 3 模型 × 3 条件
2026-04-05 09:31:59,392 [INFO] 已完成: 1290 条
2026-04-05 09:31:59,406 [INFO] >>> professional_medium->professional_strong, k=5, model=gpt-4o-mini
2026-04-05 09:31:59,410 [INFO] >>> professional_medium->professional_strong, k=5, model=claude-3-haiku
2026-04-05 09:31:59,414 [INFO] >>> professional_medium->professional_strong, k=5, model=gemini-1.5-flash
2026-04-05 09:31:59,416 [INFO] >>> professional_medium->professional_strong, k=10, model=gpt-4o-mini
2026-04-05 09:31:59,425 [INFO] >>> professional_medium->professional_strong, k=10, model=claude-3-haiku
2026-04-05 09:31:59,425 [INFO] >>> professional_medium->professional_strong, k=10, model=gemini-1.5-flash
2026-04-05 09:31:59,428 [INFO] >>> professional_medium->professional_strong, k=20, model=gpt-4o-mini
2026-04-05 09:31:59,428 [INFO] >>> professional_medium->professional_strong, k=20, model=claude-3-haiku
2026-04-05 09:31:59,433 [INFO] >>> professional_medium->professio

## 5. 数据文件检查

In [10]:
# ====== Cell 5: 最终检查 ======
print("="*60)
print("  第一阶段完成 — 数据文件清单")
print("="*60)

for name, path in [
    ("RQ1/RQ2 响应数据", config.RESULTS_DIR / "rq1_rq2_responses.csv"),
    ("RQ3 响应数据",     config.RESULTS_DIR / "rq3_responses.csv"),
    ("人工抽检样本",     config.RESULTS_DIR / "human_spotcheck_sample.csv"),
]:
    if path.exists():
        size = path.stat().st_size / 1024
        try:
            n = len(pd.read_csv(path))
        except:
            n = "?"
        print(f"  ✓ {name}: {path.name} ({n} 行, {size:.0f} KB)")
    else:
        print(f"  ✗ {name}: 未生成")

print("\n下一步: 打开 stage2_analysis.py 进行统计分析。")
print("数据文件可以随时用 Excel/Python/R 打开做额外分析。")

  第一阶段完成 — 数据文件清单
  ✓ RQ1/RQ2 响应数据: rq1_rq2_responses.csv (53865 行, 65369 KB)
  ✓ RQ3 响应数据: rq3_responses.csv (1620 行, 1990 KB)
  ✓ 人工抽检样本: human_spotcheck_sample.csv (? 行, 53 KB)

下一步: 打开 stage2_analysis.py 进行统计分析。
数据文件可以随时用 Excel/Python/R 打开做额外分析。
